# market_analysis — Pandas 시계열

- 담당: (이름)  브랜치: `<본인 github id>`
- 규칙: 커밋 전 `Kernel → Restart & Clear Output`. 이 노트북은 1인 전용.


In [ ]:
import sys, numpy as np, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from backend.services.portfolio.calculator import load_data, daily_total
from backend.services.market.analyzer import load_market, vs_index, shock_days, contribution
from backend.services.market.indicators import drawdown
pf, sp = load_data(ROOT)

## 1. 일자별 총 평가금액  (SQL [B-1]/[B-2]와 동일)

In [ ]:
daily = daily_total(pf, sp)      # SQL [B-1]/[B-2]와 동일
daily

## 2. 변화가 가장 컸던 날 / 주간 집계 / 최대 낙폭

In [ ]:
biggest_up = daily.nlargest(5, "diff_pct")
biggest_down = daily.nsmallest(5, "diff_pct")
weekly = daily.set_index("trade_date")["total_eval"].resample("W").agg(["first","last","min","max"])
weekly["change_pct"] = ((weekly["last"]-weekly["first"])/weekly["first"]*100).round(2)
daily["drawdown_pct"] = drawdown(daily["total_eval"])
biggest_up, biggest_down, weekly.head()

## 3. 시각화 (선택)

In [ ]:
ax = daily.plot(x="trade_date", y="total_eval", figsize=(10,4), title="Portfolio total value")

## 4. 시장지수 비교 (SQL [B-6]/[B-7]/[B-8]과 동일)

In [ ]:
mi = load_market(ROOT)                       # data/market/index.csv
vs = vs_index(daily, mi)                     # SQL [B-6]과 동일 (첫 번째 지수)
print("거래일 불일치:", len(set(daily.trade_date) ^ set(mi.trade_date)))
vs

In [ ]:
shock = shock_days(vs, 5)                    # SQL [B-7]
hold = pf.groupby("ticker", as_index=False)["quantity"].sum()
contrib = contribution(sp, hold, shock["trade_date"])   # SQL [B-8]
shock, contrib.groupby("trade_date").head(3)

## 5. 초과수익 최대/최소 구간, 상대강도 추이

In [ ]:
best, worst = vs.nlargest(3,"excess_pct"), vs.nsmallest(3,"excess_pct")
ax = vs.plot(x="trade_date", y="rel_strength", figsize=(10,3), title="Portfolio / index (relative strength)")
best, worst

## 6. 결과 내보내기

In [ ]:
out = ROOT/"outputs"; out.mkdir(exist_ok=True)
daily.to_csv(out/"pandas_D_daily_total.csv", index=False)
weekly.reset_index().to_csv(out/"pandas_D_weekly.csv", index=False)
vs.to_csv(out/"pandas_D_vs_index.csv", index=False)
shock.to_csv(out/"pandas_D_index_shock_days.csv", index=False)
contrib.to_csv(out/"pandas_D_contribution.csv", index=False)